# GCSE Helper – Exercise → Structured Help JSON (Text or Image)

This notebook demonstrates an end-to-end *development* workflow:

1. Provide an exercise as **typed text** *or* an **image** (screenshot/photo).
2. Extract text (OCR) from image **if possible**.
3. Normalize + hash the exercise for caching.
4. If not cached, call **OpenAI** to generate the structured JSON output (tiers: nudge/hint/steps/worked/teachback).
5. Validate the JSON (lightweight checks) and store in a local cache (simulating DynamoDB).
6. Display the sections your UI would render.

## Notes
- Set `OPENAI_API_KEY` in your environment.
- OCR is implemented via `pytesseract` **if installed**; otherwise the notebook falls back to a manual text entry.
- The cache here is an in-memory dict + optional JSON file on disk. Replace with DynamoDB later.


In [16]:
# ---- Optional OCR setup ----
# The notebook will try to use pytesseract if available.
# If it's not installed, you can still supply the exercise as typed text.

OCR_AVAILABLE = False
try:
    import pytesseract
    from PIL import Image
    OCR_AVAILABLE = True
    print('OCR: pytesseract available ✅')
except Exception as e:
    print('OCR: pytesseract not available (that\'s OK).')
    print('Reason:', type(e).__name__, str(e)[:200])


OCR: pytesseract available ✅


In [1]:
# ---- Config ----
import os, re, json, hashlib
from datetime import datetime, timezone
from pathlib import Path

OPENAI_MODEL = os.environ.get('OPENAI_MODEL', 'gpt-4.1-mini')  # adjust if you like
CACHE_PATH = Path('./gcse_cache.json')

print('Model:', OPENAI_MODEL)
print('Cache file:', CACHE_PATH.resolve())


Model: gpt-4.1-mini
Cache file: /Users/simonneumann/code/GCSE_AI_agent/backend/scripts/gcse_cache.json


In [30]:
# ---- Load / save local cache (simulating DynamoDB) ----

def load_cache(path: Path) -> dict:
    if path.exists():
        try:
            return json.loads(path.read_text(encoding='utf-8'))
        except Exception:
            return {}
    return {}

def save_cache(path: Path, cache: dict) -> None:
    path.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding='utf-8')

CACHE = load_cache(CACHE_PATH)
print('Cached items:', len(CACHE))


Cached items: 1


## 1) Helpers: normalization + hashing + light validation

We create a stable key from the normalized prompt so we can reuse cached results.


In [18]:
def normalize_exercise_text(text: str) -> str:
    """Normalize text for hashing + caching.
    Keep it conservative: normalize whitespace, strip, standardize unicode minus.
    """
    t = text.replace('−', '-')
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def exercise_hash(normalized_text: str, schema_version: str = '1.0.0') -> str:
    payload = f"{schema_version}||{normalized_text}".encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def light_validate_response(obj: dict) -> None:
    """Lightweight validation (you can replace with jsonschema or pydantic later)."""
    required_top = ['schema_version', 'request', 'exercise', 'analysis', 'help']
    for k in required_top:
        if k not in obj:
            raise ValueError(f"Missing top-level field: {k}")

    # Check tiers exist
    tiers = obj.get('help', {}).get('tiers', {})
    for tier in ['nudge', 'hint', 'steps', 'worked', 'teachback']:
        if tier not in tiers:
            raise ValueError(f"Missing help.tiers.{tier}")
        if not isinstance(tiers[tier].get('content', []), list) or len(tiers[tier]['content']) == 0:
            raise ValueError(f"help.tiers.{tier}.content must be a non-empty list")

    # Must include normalized_text
    nt = obj.get('exercise', {}).get('prompt', {}).get('normalized_text')
    if not nt or not isinstance(nt, str):
        raise ValueError('exercise.prompt.normalized_text missing or invalid')


## 2) OCR extraction (optional)

If you pass an image path and OCR is available, we try to extract the text.


In [19]:
def extract_text_from_image(image_path: str) -> dict:
    """Return {raw_text, normalized_text, extraction:{status, confidence, ambiguities}}"""
    if not OCR_AVAILABLE:
        return {
            'raw_text': '',
            'normalized_text': '',
            'extraction': {
                'status': 'failed',
                'confidence': 0.0,
                'ambiguities': [{'message': 'OCR not available. Please type the exercise text.'}]
            }
        }

    img = Image.open(image_path)
    raw = pytesseract.image_to_string(img)
    norm = normalize_exercise_text(raw)

    # Very naive confidence heuristic (you can improve):
    confidence = 0.9 if len(norm) >= 5 else 0.2
    status = 'ok' if confidence >= 0.6 else 'needs_confirmation'

    ambiguities = []
    if status != 'ok':
        ambiguities.append({'message': 'Extracted text looks short/uncertain. Please confirm or edit.'})

    return {
        'raw_text': raw,
        'normalized_text': norm,
        'extraction': {
            'status': status,
            'confidence': float(confidence),
            'ambiguities': ambiguities
        }
    }


## 3) OpenAI call to generate the structured JSON

This uses the modern `openai` Python library (`OpenAI()` client). If it isn't installed, run the install cell.


In [20]:
# If needed, install dependencies (uncomment and run)
!pip -q install "openai>=1.0.0" "jsonschema>=4.22" pillow pytesseract



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
def openai_generate_help_json(
    *,
    normalized_text: str,
    origin_type: str = 'student_homework',
    origin_label: str = 'Student homework',
    schema_version: str = '1.0.0',
    year_group: int | None = 9,
    tier: str = 'unknown',
    desired_help_level: str = 'auto',
) -> dict:
    """Call OpenAI and request strict JSON output."""
    api_key = os.environ.get('OPENAI_API_KEY')

    if not api_key:
        raise RuntimeError('OPENAI_API_KEY is not set in the environment')

    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    request_id = f"req_{hashlib.sha1((normalized_text+now_iso()).encode()).hexdigest()[:12]}"
    exercise_id = f"ex_{hashlib.sha1(normalized_text.encode()).hexdigest()[:12]}"

    system = (
        "You are a GCSE tutor assistant. Return ONLY valid JSON that conforms to the requested structure. "
        "Use UK tone (en-GB) and age-appropriate wording for Year 9 unless otherwise specified. "
        "Do not include markdown fences. Do not include commentary."
    )

    user = {
        "schema_version": schema_version,
        "request": {
            "id": request_id,
            "created_at": now_iso(),
            "locale": "en-GB",
            "student_context": {
                "year_group": year_group,
                "tier": tier,
                "desired_help_level": desired_help_level,
                "what_ive_tried": ""
            },
            "input": {"modality": "typed_text", "typed_text": {"text": normalized_text}}
        },
        "exercise": {
            "exercise_id": exercise_id,
            "origin": {"type": origin_type, "label": origin_label, "created_at": now_iso()},
            "prompt": {"normalized_text": normalized_text, "raw_text": normalized_text, "attachments": []},
            "extraction": {"status": "ok", "confidence": 1, "ambiguities": []}
        },
        "analysis": {
            "subject": "maths",
            "topics": [],
            "difficulty": {"gcse_tier_hint": "unknown", "confidence": 0.5},
            "prerequisites": [],
            "common_mistakes": []
        },
        "help": {
            "recommended_start": "nudge",
            "tiers": {
                "nudge": {"title": "", "content": [{"type": "plain", "text": ""}]},
                "hint": {"title": "", "content": [{"type": "plain", "text": ""}]},
                "steps": {"title": "", "content": [{"type": "plain", "text": ""}]},
                "worked": {"title": "", "content": [{"type": "plain", "text": ""}]},
                "teachback": {"title": "", "content": [{"type": "plain", "text": ""}]}
            },
            "formulas_used": [],
            "check_your_answer": {"instruction": "", "worked_check": ""},
            "practice": []
        }
    }

    prompt = (
        "Fill in and complete the JSON object below. Keep the same keys. Replace placeholder empty strings/arrays with real content. "
        "Populate analysis.topics, analysis.prerequisites, analysis.common_mistakes appropriately. "
        "Populate help.tiers with progressive help: nudge (1-2 lines), hint (bullets), steps (numbered bullets), worked (math lines), teachback (why it works). "
        "Use help.formulas_used only if relevant. Include a check_your_answer with substitution check. Add 2-3 practice questions with final answers only. "
        "Return strictly valid JSON.\n\n"
        + json.dumps(user, ensure_ascii=False)
    )

    chat = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
        max_tokens=1200,
    )

    text = (chat.choices[0].message.content or "").strip()

    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        repair = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": system},
                {
                    "role": "user",
                    "content": "Fix and return ONLY valid JSON for the following (no commentary):\n" + text,
                },
            ],
            temperature=0,
            max_tokens=1200,
        )
        obj = json.loads((repair.choices[0].message.content or "").strip())

    light_validate_response(obj)
    return obj

## 4) Main function: provide text or image, then cache and return JSON


In [22]:
def generate_or_get_cached(
    *,
    exercise_text: str | None = None,
    image_path: str | None = None,
    origin_type: str = 'student_homework',
    origin_label: str = 'Student homework',
    schema_version: str = '1.0.0'
) -> dict:
    if not exercise_text and not image_path:
        raise ValueError('Provide either exercise_text or image_path')

    raw_text = ''
    extraction = {"status": "ok", "confidence": 1, "ambiguities": []}

    if image_path:
        ocr = extract_text_from_image(image_path)
        raw_text = ocr['raw_text']
        extraction = ocr['extraction']
        normalized_text = ocr['normalized_text']
        if extraction['status'] != 'ok':
            # Return early so UI can ask for confirmation
            return {
                "schema_version": schema_version,
                "exercise": {
                    "origin": {"type": origin_type, "label": origin_label},
                    "prompt": {"normalized_text": normalized_text or '', "raw_text": raw_text or ''},
                    "extraction": extraction
                }
            }
    else:
        raw_text = exercise_text or ''
        normalized_text = normalize_exercise_text(raw_text)

    key = exercise_hash(normalized_text, schema_version=schema_version)
    if key in CACHE:
        print('Cache hit ✅')
        return CACHE[key]

    print('Cache miss — calling OpenAI…')
    obj = openai_generate_help_json(
        normalized_text=normalized_text,
        origin_type=origin_type,
        origin_label=origin_label,
        schema_version=schema_version,
        year_group=9
    )

    # preserve extraction info if we got it from OCR
    try:
        obj.setdefault('exercise', {}).setdefault('prompt', {})['raw_text'] = raw_text
        obj.setdefault('exercise', {})['extraction'] = extraction
    except Exception:
        pass

    CACHE[key] = obj
    save_cache(CACHE_PATH, CACHE)
    return obj


## 5) Try it with typed text


In [31]:
result = generate_or_get_cached(exercise_text='Solve for x: 5x + 7 = 17', origin_type='student_homework')
print(json.dumps(result, indent=2, ensure_ascii=False)[:2000])


Cache hit ✅
{
  "schema_version": "1.0.0",
  "request": {
    "id": "req_31ef2c46d0fb",
    "created_at": "2025-12-29T14:35:19.337527+00:00",
    "locale": "en-GB",
    "student_context": {
      "year_group": 9,
      "tier": "unknown",
      "desired_help_level": "auto",
      "what_ive_tried": ""
    },
    "input": {
      "modality": "typed_text",
      "typed_text": {
        "text": "Solve for x: 5x + 7 = 17"
      }
    }
  },
  "exercise": {
    "exercise_id": "ex_64ca330a25ca",
    "origin": {
      "type": "student_homework",
      "label": "Student homework",
      "created_at": "2025-12-29T14:35:19.337531+00:00"
    },
    "prompt": {
      "normalized_text": "Solve for x: 5x + 7 = 17",
      "raw_text": "Solve for x: 5x + 7 = 17",
      "attachments": []
    },
    "extraction": {
      "status": "ok",
      "confidence": 1,
      "ambiguities": []
    }
  },
  "analysis": {
    "subject": "maths",
    "topics": [
      "linear equations",
      "algebraic manipulation",


## 6) Render-like preview (what the UI would show)


In [24]:
def preview_tier(obj: dict, tier: str):
    t = obj['help']['tiers'][tier]
    print(f"\n=== {tier.upper()} — {t.get('title','')} ===")
    for block in t['content']:
        prefix = {'plain':'', 'math':'', 'bullet':'- ', 'warning':'⚠️ '}.get(block['type'], '')
        print(prefix + block['text'])

for tier in ['nudge','hint','steps','worked','teachback']:
    preview_tier(result, tier)

print('\nCheck:', result['help'].get('check_your_answer', {}).get('worked_check',''))
print('\nPractice:')
for p in result['help'].get('practice', []):
    print('-', p['question'], '→', p['answer'])



=== NUDGE — Try isolating the variable ===
Think about how you can get x on its own on one side of the equation.

=== HINT — Remember inverse operations ===
- To undo adding 7, what should you do?
- After that, how do you undo multiplying by 5?

=== STEPS — Step-by-step solution ===
1. Subtract 7 from both sides of the equation: 5x + 7 - 7 = 17 - 7
2. Simplify both sides: 5x = 10
3. Divide both sides by 5 to isolate x: (5x)/5 = 10/5
4. Simplify the division: x = 2

=== WORKED — Worked solution ===
5x + 7 = 17
5x + 7 - 7 = 17 - 7
5x = 10
5x ÷ 5 = 10 ÷ 5
x = 2

=== TEACHBACK — Why this method works ===
We use inverse operations to keep the equation balanced while isolating x.
Subtracting 7 removes the constant term on the left side.
Dividing by 5 reverses the multiplication, leaving x alone.

Check: 5(2) + 7 = 10 + 7 = 17, which matches the right side, so x = 2 is correct.

Practice:
- Solve for x: 3x + 4 = 19 → x = 5
- Solve for x: 7x - 3 = 25 → x = 4
- Solve for x: 4x + 9 = 29 → x = 5

## 7) Try it with an image (OCR)

Put a screenshot/photo path in `image_path`.

If OCR isn't available, the function will return `extraction.status = failed` and ask you to type the text.


In [25]:
# Example:
# image_path = 'my_screenshot.png'
# result_img = generate_or_get_cached(image_path=image_path, origin_type='student_homework')
# print(json.dumps(result_img, indent=2, ensure_ascii=False)[:2000])
